# 02 · FireNet-CNN - 5-fold cross-validation
Architecture (verified to give **2,753,217 parameters**, as in the paper): 5 × [Conv3×3 *valid* → ReLU → MaxPool2×2 → BatchNorm] with 32/64/128/256/512 filters → Flatten (2×2×512) → Dense512 → Dropout.5 → Dense256 → Dropout.5 → Dense1 sigmoid. Adam 1e-4, BCE, batch 32, 50 epochs.

Two protocols are run:
* **`paper`** - synthetic images are pooled with the real ones *before* the 5-fold split (what the paper describes).
* **`strict`** - folds are built from **real images only**; all synthetic images are added to the training side only. This is the leakage-free number.

In [ ]:
# ---- Colab setup (run first; if you ever see Keras-3 errors: Runtime > Restart session, then re-run) ----
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"          # the paper's code style needs Keras 2
!pip -q install tf_keras
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/firenet"       # everything lives here, so a disconnect loses nothing
sys.path.insert(0, PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from types import SimpleNamespace
from IPython.display import Image, display
import firenet as fn                             # written to Drive by 00_setup_and_data.ipynb
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
DATASETS = ["A", "B"]          # which real dataset(s)
MODES    = ["paper", "strict"] # paper = split AFTER adding synthetic (as in the paper); strict = synthetic only in training folds
EPOCHS   = 2                   # <-- smoke test first! then set to 50 for the real run

cfg = SimpleNamespace(data=f"{PROJECT}/data", dataset="B", mode="paper", no_synth=False,
                      epochs=EPOCHS, bs=32, lr=1e-4, seed=42, out=f"{PROJECT}/runs")
def get(d):
    return fn.load_cached(cfg.data, d, use_synth=True, cache_dir=f"{PROJECT}/cache")

In [ ]:
m = fn.build_firenet(); m.summary()
assert m.count_params() == 2_753_217, m.count_params()
print("parameter count matches the paper")

## Run (start with `EPOCHS = 2` as a smoke test, then set 50)

In [ ]:
results = {}
for d in DATASETS:
    cfg.dataset = d
    X, y, s = get(d)
    for mode in MODES:
        cfg.mode = mode
        results[(d, mode)] = fn.run_cv(X, y, s, fn.build_firenet, cfg, "firenet")
pd.DataFrame(results).T.round(4)

## Figures (Fig. 12-25 of the paper): curves, confusion matrices, ROC

In [ ]:
d, mode = DATASETS[-1], MODES[0]      # change to inspect other runs
run = f"{PROJECT}/runs/{d}_{mode}_firenet"
for f in ("accuracy_per_fold", "loss_per_fold", "accuracy_all_folds", "loss_all_folds", "confusion", "roc"):
    display(Image(f"{run}/firenet_{f}.png"))
pd.read_csv(f"{run}/fold_results.csv").round(4)